# Konsolidierung der EMR-Ist-Datenbasis

Dieses Notebook baut aus den EMR-Rohdaten der SPEDION-Fahrer-App eine bereinigte Ist-Basis und speichert sie als CSV und Pickle. Wie beim Soll folgt es der einheitlichen Konsolidierungs-Struktur. Als erstes die Daten einlesen, auf März filtern, Mafi entfernen, Fahrer und Kennzeichen normalisieren, Dubletten entfernen, exportieren. Anders als das Soll liegen die EMR-Daten auf Meldungsebene vor, also als einzelne GPS- und Statusmeldungen ohne Tour-Zuordnung

## EMR-Rohdaten einlesen und konsolidieren

Die Ist-Daten liegen als fünf Einzelmeldungsreports vor, je eine Datei pro Kalenderwoche (02.03. bis 05.04.2026). Jede Datei enthält ein Zusammenfassungs-Sheet sowie je ein Detail-Sheet pro Fahrzeug. Das Zusammenfassungs-Sheet überspringen wir bewusst, da es nur eine aggregierte Wochenübersicht enthält und keine einzelnen Meldungen Wir greifen deshalb gezielt nur die Detail-Sheets ab, erkennbar an der Endung "- Fahrzeugdetails" im Sheet-Namen. In den Detail-Sheets beginnen die Meldungen ab Zeile 9, die Kopfzeile steht in Zeile 8. Das Kennzeichen steht nur im Sheet-Namen, daher übernehmen wir es als eigene Spalte QUELLE_KENNZEICHEN. Alle Detail-Sheets aller fünf Wochen werden zu einem durchgehenden DataFrame emr_raw zusammengeführt.

In [18]:
import pandas as pd
from pathlib import Path
import re

# Anzeigeoptionen, damit DataFrames vollständig sichtbar sind
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", None)

BASE_PATH = Path.cwd().parent / "data" / "raw"

# Alle EMR-Wochendateien (je Datei = eine Kalenderwoche)
emr_dateien = sorted(BASE_PATH.glob("Einzelmeldungsreport_*.xlsx"))

teile = []
for datei in emr_dateien:
    xl = pd.ExcelFile(datei)
    detail_sheets = [s for s in xl.sheet_names if s.endswith("- Fahrzeugdetails")]
    for sheet in detail_sheets:
        teil = pd.read_excel(datei, sheet_name=sheet, header=8)  # Kopfzeile in Zeile 8
        teil["QUELLE_KENNZEICHEN"] = sheet.replace(" - Fahrzeugdetails", "")
        teil["QUELLE_DATEI"] = datei.name
        teile.append(teil)

emr_raw = pd.concat(teile, ignore_index=True)
print(f"Dateien: {len(emr_dateien)} | Zeilen: {len(emr_raw)}")
pd.concat([emr_raw.head(), emr_raw.tail()])

Dateien: 5 | Zeilen: 188720


,Fahrer PIN,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Richtung,Geschwindigkeit,KM-Stand,Gesamtverbrauch,Tankfüllstand,Breitengrad,Längengrad,Position,Meldungsquelle,QUELLE_KENNZEICHEN,QUELLE_DATEI
0,NaN,,1332,Tour im Fahrzeug angekommen,2026-03-02 12:52:19,180,1,0.000,0.0,0.0,"53,52311","9,98774","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 12",SPEDION App,Mafi,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-12_27_05....
1,NaN,,1382,TourDeleteByID - OK,2026-03-02 12:52:20,0,0,0.000,0.0,0.0,"53,52297","9,98831","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 17",SPEDION App,Mafi,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-12_27_05....
2,NaN,,1344,"Tour, Place oder Order im Fahrzeug gelöscht",2026-03-02 12:52:20,0,0,0.000,0.0,0.0,"53,52297","9,98831","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 17",SPEDION App,Mafi,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-12_27_05....
3,NaN,,1332,Tour im Fahrzeug angekommen,2026-03-02 12:52:21,0,0,0.000,0.0,0.0,"53,52298","9,98828","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 17",SPEDION App,Mafi,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-12_27_05....
4,NaN,,2100,SPEDION Mobile Control Nutzung,2026-03-02 12:52:25,0,0,0.000,0.0,0.0,"53,523","9,98832","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 17",SPEDION App,Mafi,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-12_27_05....
188715,NaN,,30007,Shutdown - Generic Third Party Telematic Device Position...,2026-04-04 05:44:04,0,0,809598.055,0.0,0.0,"53,52338","9,9862","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 12",TruckBox,WL-PL431,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-13_52_32....
188716,NaN,,30023,Startup on Timer,2026-04-05 02:00:34,0,0,809598.055,0.0,0.0,"53,52338","9,9862","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 12",TruckBox,WL-PL431,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-13_52_32....
188717,NaN,,1615,Tägliche Meldung,2026-04-05 02:00:34,0,0,809598.055,0.0,0.0,"53,52338","9,9862","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 12",TruckBox,WL-PL431,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-13_52_32....
188718,NaN,,30034,Configuration Hash,2026-04-05 02:01:36,0,0,809598.055,0.0,0.0,"53,52327","9,98605","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 12",TruckBox,WL-PL431,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-13_52_32....
188719,NaN,,30007,Shutdown - Generic Third Party Telematic Device Position...,2026-04-05 02:07:23,0,0,809598.055,0.0,0.0,"53,52334","9,98607","DE - 20457 Hamburg (Hamburg-Mitte), Brandenburger Straße 12",TruckBox,WL-PL431,Einzelmeldungsreport_HTS-Cargo_GmbH_2026_04_14-13_52_32....


## Untersuchung und Aufbereitung der EMR-Daten

Wir starten mit info(), um Datentypen und Befüllung der 16 Spalten zu sehen. Daraus ergeben sich, dass Breitengrad und Längengrad als Text vorliegen und in Zahlen umgewandelt werden müssen. Fahrer PIN ist nur teilweise gefüllt und wird ebenfalls nochmal genauer untersucht.

In [19]:
emr_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 188720 entries, 0 to 188719
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   Fahrer PIN            165978 non-null  float64       
 1   Fahrername            188720 non-null  str           
 2   Formularnummer        188720 non-null  int64         
 3   Formularbeschreibung  188720 non-null  str           
 4   Meldungszeit          188720 non-null  datetime64[us]
 5   Richtung              188720 non-null  int64         
 6   Geschwindigkeit       188720 non-null  int64         
 7   KM-Stand              188720 non-null  float64       
 8   Gesamtverbrauch       188720 non-null  float64       
 9   Tankfüllstand         188720 non-null  float64       
 10  Breitengrad           188720 non-null  str           
 11  Längengrad            188720 non-null  str           
 12  Position              188720 non-null  str           
 13  Meldungsqu

## Koordinaten in Zahlen umwandeln

Breitengrad und Längengrad kommen als Komma-Text und werden zu float, damit sie für die spätere Geofence- und Routenanalyse nutzbar sind. describe() am Ende zeigt, dass der Wertebereich zu Norddeutschland passt und anders als bei den Soll-Daten gibt es hier keine Null-Koordinaten.

In [20]:
# Breitengrad/Längengrad in float umwandeln
for spalte in ["Breitengrad", "Längengrad"]:
    emr_raw[spalte] = pd.to_numeric(emr_raw[spalte].str.replace(",", ".", regex=False))

emr_raw[["Breitengrad", "Längengrad"]].describe()

,Breitengrad,Längengrad
count,188720.000000,188720.000000
mean,53.547536,9.966510
std,0.203165,0.361147
min,51.298260,8.152620
25%,53.520570,9.964250
50%,53.523230,9.986065
75%,53.585160,9.989430
max,54.781510,13.649580


## Auf den Analysezeitraum März beschränken

Das tail() am Anfang hat gezeigt, dass die EMR-Wochen bis zum 5. April reichen. Wie beim Soll begrenzen wir auf März 2026 mit denselben Grenzen (>= 2026-03-01 und < 2026-04-01), damit alle drei Quellen denselben Zeitraum abdecken. Wir filtern über die Meldungszeit, so fallen die April-Zeilen weg und der Rest bleibt vollständig.

In [21]:
vorher = len(emr_raw)
maerz = (emr_raw["Meldungszeit"] >= "2026-03-01") & (emr_raw["Meldungszeit"] < "2026-04-01")
emr_raw = emr_raw[maerz].copy()
print(f"Außerhalb März entfernt: {vorher - len(emr_raw)} Zeilen | verbleibend: {len(emr_raw)}")

Außerhalb März entfernt: 16764 Zeilen | verbleibend: 171956


nunique() gibt einen Kardinalitäts-Überblick und trennt die hochkardinalen Spalten (Zeitstempel, Koordinaten, KM-Stand) von den wenigen kategorialen, die wir als Nächstes über ihre Verteilung ansehen.

In [22]:
emr_raw.nunique()

Fahrer PIN                  14
Fahrername                  15
Formularnummer              77
Formularbeschreibung        76
Meldungszeit            142463
Richtung                   361
Geschwindigkeit             99
KM-Stand                 48880
Gesamtverbrauch          14462
Tankfüllstand              247
Breitengrad              24394
Längengrad               31958
Position                  6877
Meldungsquelle               2
QUELLE_KENNZEICHEN          13
QUELLE_DATEI                 5
dtype: int64

Die Verteilung der kategorialen Spalten zeigt, dass in QUELLE_KENNZEICHEN wieder das Fahrzeug Mafi auftaucht, das im nächsten Schritt entfernt wird. Die Formularbeschreibung listet die Meldungstypen im Klartext und Meldungsquelle hat nur zwei Werte und ist damit ein Kandidat für die abschließende Reduktion.

In [23]:
# Verteilung der kategorialen Spalten
for spalte in ["QUELLE_KENNZEICHEN", "Formularbeschreibung", "Meldungsquelle"]:
    display(emr_raw[spalte].value_counts(dropna=False))

QUELLE_KENNZEICHEN
Mafi          20989
PI-EN 444     17872
Plö-TS853     16794
TS859         16387
OD-TS 8041    14518
OD-TS 8046    13966
OD-TS 8050    13906
Plö-TS856     13321
WL-PL431      12143
OD-TS 8048    10469
OD-TS 8044     9187
PI-EN 900      7055
Plo-TS857      5349
Name: count, dtype: int64

Formularbeschreibung
Rollen                                               43173
Beginn Rollen                                        34188
Ende Rollen                                          34162
Anhalten                                             23515
Place wurde aktiviert                                10039
                                                     ...  
Tankfüllstand Event - Große Abnahme                      1
Support Info mit Anhang                                  1
Android Update installed by MDM successful               1
Warten                                                   1
Storno (Tour, Place oder Order) nicht verarbeitet        1
Name: count, Length: 76, dtype: int64

Meldungsquelle
SPEDION App    92711
TruckBox       79245
Name: count, dtype: int64

## Mafi entfernen

Wir entfernen alle 20.989 Mafi-Zeilen über QUELLE_KENNZEICHEN und eine manuelle Überprüfung der EMR-Daten zeigt, dass das Mafi-Fahrzeug von zwei Fahrern gefahren wird. Das heißt Mirzaie fällt  komplett weg, weil er ausschließlich Mafi gefahren ist. Stoffer bleibt erhalten, weil er neben seinen Mafi-Fahrten auch dem Fahrzeug WL-PL431 zugeteilt ist.

In [24]:
vorher = len(emr_raw)
emr_raw = emr_raw[emr_raw["QUELLE_KENNZEICHEN"] != "Mafi"].copy()
print(f"Mafi entfernt: {vorher - len(emr_raw)} Zeilen | verbleibend: {len(emr_raw)}")

Mafi entfernt: 20989 Zeilen | verbleibend: 150967


Weil info() die Lücke in Fahrer PIN gezeigt hat, sehen wir uns beide Fahrer-Spalten über ihre Häufigkeiten an. Auffällig ist, dass der häufigste Fahrername ein leerer Eintrag ist, also Leerzeichen, kein NaN, weshalb info() ihn nicht als fehlend zählt und Fahrer PIN hat genau gleich viele NaN-Werte. Beide Spalten tragen also dieselbe Information mit derselben Lücke, einmal als Code, einmal als Klartext.

In [25]:
# Fahrer-Spalten über ihre Häufigkeiten ansehen
for spalte in ["Fahrername", "Fahrer PIN"]:
    display(emr_raw[spalte].value_counts(dropna=False).head(10))

Fahrername
                            20786
Krüger, Alexander           16021
Jorczik, Christian          14658
Hruszka, Krzysztof          14057
Calina, Florin Alexandru    13646
Hesse, Sven-Erik            13131
Giewon, Mariusz             12893
Szmajdewicz, Marcin         11823
Oelschläger, Marcel          9471
Borrs, Thomas                8436
Name: count, dtype: int64

Fahrer PIN
NaN       20786
8180.0    16021
8160.0    14658
8197.0    14057
8168.0    13646
8317.0    13131
8310.0    12893
8311.0    11823
8258.0     9471
8205.0     8436
Name: count, dtype: int64

## Fahrernamen angleichen und nicht benötigte Spalten entfernen

Als Nächstes bringen wir die Fahrernamen auf den Stand der Soll-Daten, also leere Einträge zu NaN, Umlaute ausgeschrieben (Krüger zu Krueger, Oelschläger zu Oelschlaeger) und zwei abweichende Schreibweisen derselben Personen an die Soll-Schreibweise angeglichen (Hruszka, Krzysztof zu Hrzuska, Krzystof und Teme, Abdoulaeye zu Teme, Abdoulaye). Die Soll-Form ist die Referenz für die Namen, an der sich beide Ist-Quellen ausrichten, da sie als erstes behandelt wurde. Danach werden die Spalten entfernt, die keinerlei Relevanz für die Projektziele haben wie Fahrer PIN (redundant zum Namen), Gesamtverbrauch und Tankfüllstand sowie Richtung (Ist für die Tour-Rekonstruktion bereits durch die Koordinaten abgedeckt).

In [26]:
# Fahrername auf das Soll-Schema bringen
umlaut_expand = str.maketrans({"ä": "ae", "ö": "oe", "ü": "ue", "Ä": "Ae", "Ö": "Oe", "Ü": "Ue", "ß": "ss"})
namens_korrektur = {
    "Hruszka, Krzysztof": "Hrzuska, Krzystof",   
    "Teme, Abdoulaeye": "Teme, Abdoulaye",
}
emr_raw["Fahrername"] = (
    emr_raw["Fahrername"]
    .str.strip().replace("", pd.NA)    
    .str.translate(umlaut_expand)      
    .replace(namens_korrektur)         
)

emr_raw = emr_raw.drop(columns=["Fahrer PIN", "Gesamtverbrauch", "Tankfüllstand", "Richtung"])

print("Fahrername fehlend:", emr_raw["Fahrername"].isna().sum())
print("Spalten verbleibend:", emr_raw.shape[1])

Fahrername fehlend: 20786
Spalten verbleibend: 12


Ohne die Kraftstoff- und Richtungsspalten ist describe() jetzt aussagekräftiger. Bei Geschwindigkeit und KM-Stand fällt jeweils ein hoher Null-Anteil auf, der als Nächstes untersucht wird. Formularnummer und die Koordinaten haben keine statistische Aussage.

In [27]:
emr_raw.describe()

,Formularnummer,Meldungszeit,Geschwindigkeit,KM-Stand,Breitengrad,Längengrad
count,150967.000000,150967,150967.000000,150967.000000,150967.000000,150967.000000
mean,16551.620076,2026-03-16 05:44:03.360780,4.230256,281596.023097,53.553546,9.961703
min,1156.000000,2026-03-02 00:00:00,0.000000,0.000000,51.298260,8.152620
25%,1643.000000,2026-03-09 11:38:19,0.000000,143998.877500,53.519240,9.956760
50%,30001.000000,2026-03-16 11:27:45,3.000000,196155.845000,53.523530,9.985270
75%,30019.000000,2026-03-24 07:24:28,5.000000,462368.745000,53.600120,9.992080
max,30039.000000,2026-03-31 23:56:00,116.000000,808523.005000,54.781510,13.649580
std,14107.458261,NaN,6.411984,225662.537494,0.212679,0.363716


Wir prüfen, was hinter den Nullen steckt. Geschwindigkeit == 0 gehört zu Steh-Meldungen wie Anhalten oder Zündung an/aus, ist also kein Defekt, sondern das Stillstands-Signal, das wir für die Standzeiten brauchen. KM-Stand == 0 verteilt sich dagegen auf beide Meldungsquellen und lässt sich nicht erklären, daher wird nichts angepasst.

In [28]:
# Welche Meldungsquellen haben die meisten Geschwindigkeits-0-Meldungen?
display(emr_raw.loc[emr_raw["Geschwindigkeit"] == 0, "Formularbeschreibung"].value_counts().head())
# Welche Meldungsquellen haben die meisten KM-Stand-0-Meldungen?
display(emr_raw.loc[emr_raw["KM-Stand"] == 0, "Meldungsquelle"].value_counts())

Formularbeschreibung
Anhalten                 22760
Place wurde aktiviert     8118
Ende Rollen               2765
Zündung an                2434
Zündung aus               2422
Name: count, dtype: int64

Meldungsquelle
SPEDION App    16066
TruckBox       15264
Name: count, dtype: int64

In [29]:
df_istdaten_emr = emr_raw.drop(columns=["Meldungsquelle", "QUELLE_DATEI"])
print(df_istdaten_emr.shape)
df_istdaten_emr.head()

(150967, 10)


,Fahrername,Formularnummer,Formularbeschreibung,Meldungszeit,Geschwindigkeit,KM-Stand,Breitengrad,Längengrad,Position,QUELLE_KENNZEICHEN
2678,NaN,30012,Reboot - Generic Third Party Telematic Device Position M...,2026-03-02 00:22:52,0,314612.18,53.66768,9.99212,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS 8041
2679,NaN,30006,Startup - Generic Third Party Telematic Device Position ...,2026-03-02 00:24:00,0,314612.18,53.66768,9.99212,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS 8041
2680,NaN,1615,Tägliche Meldung,2026-03-02 01:00:00,0,314612.18,53.66759,9.99222,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS 8041
2681,NaN,30012,Reboot - Generic Third Party Telematic Device Position M...,2026-03-02 01:22:28,0,314612.18,53.66760,9.99222,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS 8041
2682,NaN,30006,Startup - Generic Third Party Telematic Device Position ...,2026-03-02 01:23:32,0,314612.18,53.66760,9.99222,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS 8041


## Kennzeichen auf das Soll-Format bringen

Auch bei den Kennzeichen wird dieselbe Normalisierung wie beim Soll angewendet, d.h. Großschreibung, Umlaute zusammenziehen, Buchstaben- und Zifferngruppen mit Bindestrich. Danach stimmen elf der zwölf Fahrzeuge mit dem Soll überein. Übrig bleibt auf jeder Seite genau ein Fahrzeug ohne Partner im Soll PLO-TS-859, in EMR TS-859. Da die anderen elf eindeutig zugeordnet sind, kann TS-859 nur dieses Soll-Fahrzeug sein. Daher wird aus TS-859 das Soll-Fahrzeug PLO-TS-859 gemacht. Das Soll-Fahrzeug RW-435 hat in EMR kein Gegenstück, weil es für dieses Fahrzeug keine Ist-Daten gibt.

In [30]:
umlaut_collapse = str.maketrans({"Ä": "A", "Ö": "O", "Ü": "U"})

def normalisiere_kennzeichen(wert):
    if pd.isna(wert):
        return wert
    wert = wert.upper().translate(umlaut_collapse)
    return "-".join(re.findall(r"[A-Z]+|[0-9]+", wert))

df_istdaten_emr["QUELLE_KENNZEICHEN"] = df_istdaten_emr["QUELLE_KENNZEICHEN"].map(normalisiere_kennzeichen)

# TS-859 und PLO-TS-859 sind dasselbe Fahrzeug
df_istdaten_emr["QUELLE_KENNZEICHEN"] = df_istdaten_emr["QUELLE_KENNZEICHEN"].replace({"TS-859": "PLO-TS-859"})

print(sorted(df_istdaten_emr["QUELLE_KENNZEICHEN"].unique()))

['OD-TS-8041', 'OD-TS-8044', 'OD-TS-8046', 'OD-TS-8048', 'OD-TS-8050', 'PI-EN-444', 'PI-EN-900', 'PLO-TS-853', 'PLO-TS-856', 'PLO-TS-857', 'PLO-TS-859', 'WL-PL-431']


## Vertiefte Datenprüfung

Vor dem Abschluss prüfen wir noch auf Dubletten, genau wie bei den Soll-Daten. Kilometerstand, Koordinaten und die Position-Spalte haben wir bereits in den Schritten zuvor angesehen, daher brauchen sie hier keine erneute Prüfung.

In [31]:
print("Exakte Voll-Dubletten:", df_istdaten_emr.duplicated().sum())
df_istdaten_emr[df_istdaten_emr.duplicated(keep=False)] \
    .sort_values(["QUELLE_KENNZEICHEN", "Meldungszeit"]) \
    [["QUELLE_KENNZEICHEN", "Meldungszeit", "Formularbeschreibung", "Geschwindigkeit", "KM-Stand"]].head(10)

Exakte Voll-Dubletten: 1848


,QUELLE_KENNZEICHEN,Meldungszeit,Formularbeschreibung,Geschwindigkeit,KM-Stand
2735,OD-TS-8041,2026-03-02 06:33:07,Place wurde aktiviert,0,314612.665
2736,OD-TS-8041,2026-03-02 06:33:07,Place wurde aktiviert,0,314612.665
2738,OD-TS-8041,2026-03-02 06:33:09,Place wurde aktiviert,0,314612.665
2739,OD-TS-8041,2026-03-02 06:33:09,Place wurde aktiviert,0,314612.665
2784,OD-TS-8041,2026-03-02 06:54:31,Place wurde aktiviert,0,314621.130
2786,OD-TS-8041,2026-03-02 06:54:31,Place wurde aktiviert,0,314621.130
2788,OD-TS-8041,2026-03-02 06:54:32,Place wurde aktiviert,0,314621.130
2789,OD-TS-8041,2026-03-02 06:54:32,Place wurde aktiviert,0,314621.130
2791,OD-TS-8041,2026-03-02 06:54:34,Place wurde aktiviert,0,314621.130
2792,OD-TS-8041,2026-03-02 06:54:34,Place wurde aktiviert,0,314621.130


Es zeigen sich 1.848 exakte Voll-Dubletten, bspw. dieselbe "Place wurde aktiviert"-Meldung doppelt zur exakt selben Sekunde, mit identischen Koordinaten und KM-Stand. Das sind echte Doppelmeldungen und keine zufällig gleichen Zeilen. Deshalb werden die exakten Dubletten entfernt, damit sie spätere Zählungen und Standzeit-Aggregationen nicht verfälschen.

In [32]:
vorher = len(df_istdaten_emr)
df_istdaten_emr = df_istdaten_emr.drop_duplicates().copy()
print(f"Exakte Dubletten entfernt: {vorher - len(df_istdaten_emr)} | verbleibend: {len(df_istdaten_emr)}")

Exakte Dubletten entfernt: 1848 | verbleibend: 149119


## Ist-Basis (EMR) abschließen und exportieren

Die bereinigte EMR-Basis wird ebenfalls in zwei Formaten gespeichert. Das Pickle wieder als Arbeitsformat für die Folge-Notebooks und die CSV als lesbare Beleg-Kopie.

In [33]:
# Gemeinsame Spaltennamen mit den Soll-Daten (für den späteren Join)
df_istdaten_emr = df_istdaten_emr.rename(columns={
    "QUELLE_KENNZEICHEN": "LKW_KENNZ",
    "Fahrername": "FAHRERNAME",
})

df_istdaten_emr_clean = df_istdaten_emr.copy()

OUT_PATH = Path.cwd().parent / "data" / "processed"
OUT_PATH.mkdir(parents=True, exist_ok=True)

ziel_pkl = OUT_PATH / "istdaten_emr_clean.pkl"
ziel_csv = OUT_PATH / "istdaten_emr_clean.csv"

# Export der bereinigten Ist-Daten als Pickle und CSV
df_istdaten_emr_clean.to_pickle(ziel_pkl)
df_istdaten_emr_clean.to_csv(ziel_csv, index=False, encoding="utf-8-sig")

print("Form:", df_istdaten_emr_clean.shape)
print(df_istdaten_emr_clean.dtypes)

Form: (149119, 10)
FAHRERNAME                         str
Formularnummer                   int64
Formularbeschreibung               str
Meldungszeit            datetime64[us]
Geschwindigkeit                  int64
KM-Stand                       float64
Breitengrad                    float64
Längengrad                     float64
Position                           str
LKW_KENNZ                          str
dtype: object


In [34]:
df_istdaten_emr_clean.head()

,FAHRERNAME,Formularnummer,Formularbeschreibung,Meldungszeit,Geschwindigkeit,KM-Stand,Breitengrad,Längengrad,Position,LKW_KENNZ
2678,NaN,30012,Reboot - Generic Third Party Telematic Device Position M...,2026-03-02 00:22:52,0,314612.18,53.66768,9.99212,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS-8041
2679,NaN,30006,Startup - Generic Third Party Telematic Device Position ...,2026-03-02 00:24:00,0,314612.18,53.66768,9.99212,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS-8041
2680,NaN,1615,Tägliche Meldung,2026-03-02 01:00:00,0,314612.18,53.66759,9.99222,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS-8041
2681,NaN,30012,Reboot - Generic Third Party Telematic Device Position M...,2026-03-02 01:22:28,0,314612.18,53.66760,9.99222,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS-8041
2682,NaN,30006,Startup - Generic Third Party Telematic Device Position ...,2026-03-02 01:23:32,0,314612.18,53.66760,9.99222,"DE - 22419 Hamburg (Langenhorn), Essener Straße 4a",OD-TS-8041
